In [1]:
from src.fetch_gtopdb_targets import fetch_targets
from src.fetch_chembl_for_targets import build_datasets
from src.sort_rarg_ligands import score_ligands_by_rarg
from pathlib import Path
import pandas as pd

In [2]:
target_urls = ["https://www.guidetopharmacology.org/GRAC/ObjectDisplayForward?objectId=590&familyId=85&familyType=NHR",
               "https://www.guidetopharmacology.org/GRAC/ObjectDisplayForward?objectId=591&familyId=85&familyType=NHR",
               "https://www.guidetopharmacology.org/GRAC/ObjectDisplayForward?objectId=592&familyId=85&familyType=NHR"]

In [3]:
fetch_targets(target_urls, output_basename='gtop_targets.tsv')

Processing target: 590
Processing target: 591
Processing target: 592
Wrote 3 entries to C:\Users\olesk\PycharmProjects\SignalScreen\data\targets\gtop_targets.tsv


,gtop_target_id,name,type,familyIds,synonyms,gene_symbol,chembl_target_url,drugbank_target_url,systematic_nomenclature
0,590,Retinoic acid receptor-&alpha;,NHR,85,"[RAR alpha 1, RAR, RAR&alpha;, retinoic acid r...",RARA,https://www.ebi.ac.uk/chembl/explore/target/CH...,http://www.drugbank.ca/biodb/polypeptides/P10276,NR1B1
1,591,Retinoic acid receptor-&beta;,NHR,85,"[HAP, HBV-activated protein, RAR beta 2, retin...",RARB,https://www.ebi.ac.uk/chembl/explore/target/CH...,http://www.drugbank.ca/biodb/polypeptides/P10826,NR1B2
2,592,Retinoic acid receptor-&gamma;,NHR,85,"[RARC, RAR&gamma;, RAR gamma 2, retinoic acid ...",RARG,https://www.ebi.ac.uk/chembl/explore/target/CH...,http://www.drugbank.ca/biodb/polypeptides/P13631,NR1B3


In [4]:
# 1. read the target TSV
targets_df = pd.read_csv(Path("data/targets/gtop_targets.tsv"), sep="\t")

# 2. extract the ChEMBL URL
target_urls = targets_df["chembl_target_url"].dropna().astype(str).tolist()

In [5]:
df_act, df_lig = build_datasets(target_urls, out_dir="data/raw")

Fetching activities for target: CHEMBL2055
 → got 1532 activity records
Failed fetch PubChem metadata for CID 76332293 HTTPSConnectionPool(host='pubchem.ncbi.nlm.nih.gov', port=443): Read timed out. (read timeout=15)
Fetching activities for target: CHEMBL2008
 → got 1436 activity records
Fetching activities for target: CHEMBL2003
 → got 1354 activity records
Wrote activities: (4322, 9) ligands: (974, 6)


In [6]:
ligands = Path.cwd() / "data" / "raw" / "ligands.tsv"
activities = Path.cwd() / "data" / "raw" / "activities.tsv"

In [7]:
scored = score_ligands_by_rarg(activities, ligands)

In [19]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
scored.to_csv(Path("data/processed") / "sorted_ligands_RARG.tsv", sep="\t", index=False)